In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib
import geopandas as gpd
import jenkspy
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
import statsmodels
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import scipy
import prophet
import warnings
warnings.filterwarnings("ignore")
from matplotlib.font_manager import FontProperties
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors
from scipy.optimize import curve_fit
import matplotlib.colors as colors
import mapclassify
from libpysal.weights.contiguity import Queen
from libpysal import examples
import geopandas as gpd
from esda.moran import Moran_Local
from splot.esda import lisa_cluster
from scipy.interpolate import UnivariateSpline
import seaborn as sns

In [ ]:
font = FontProperties(family='DejaVu Sans', weight='bold')


In [ ]:
# data = 'mp'
data = 'fb'

In [ ]:
# spatialw = 'knn'
spatialw = 'nonspatial'

In [ ]:

df_bias_mp_non = pd.read_csv('../../../data/processed/explain_bias/legacy_inputs/PH_05/Active_population_bias_nonspatial.csv', encoding="latin1")
df_bias_mp_non = df_bias_mp_non[['Area_code', 'bias']].rename(columns={'bias': 'bias_mp_non'})
df_bias_fb_non = pd.read_csv('../../../data/processed/explain_bias/legacy_inputs/TTS/Active_population_bias_nonspatial.csv', encoding="latin1")
df_bias_fb_non = df_bias_fb_non[['Area_code', 'bias']].rename(columns={'bias': 'bias_fb_non'})

df_bias = pd.merge(df_bias_mp_non, df_bias_fb_non, how='left', on = 'Area_code')

In [ ]:
df_census = pd.read_csv('../../../data/processed/explain_bias/legacy_inputs/Census_data.csv', encoding="latin1")


In [ ]:
df_census = df_census.rename(columns={'Municipality.code': 'Area_code'})

In [ ]:
df = pd.merge(df_census, df_bias, how='right', on = 'Area_code')

In [ ]:
df = df.dropna().reset_index(drop=True)

In [ ]:
df

In [ ]:
df_correlations = df[df.columns[2:-2]]
df_correlations = df_correlations.drop(range(len(df_correlations)))

df_pvalues = df[df.columns[2:-2]]
df_pvalues = df_pvalues.drop(range(len(df_pvalues)))

In [ ]:
df_correlations

In [ ]:
df_pvalues

In [ ]:
row = 0
for i in range(len(df.columns)-2, len(df.columns)):
    bias = df.columns[i]
    correlations = []
    pvalues = []
    for j in range(2, len(df.columns)-2):
        correlations.append(spearmanr(df[df.columns[j]], df[bias]).statistic)
        pvalues.append(spearmanr(df[df.columns[j]], df[bias]).pvalue)
    df_correlations.loc[row] = correlations
    df_pvalues.loc[row] = pvalues
    row += 1

In [ ]:
df_correlations = df_correlations.T
df_pvalues = df_pvalues.T

In [ ]:
df_correlations['average'] = df_correlations.mean(axis=1)
df_pvalues['average'] = df_correlations['average']

In [ ]:
df_correlations = df_correlations.sort_values(by=['average'])
df_pvalues = df_pvalues.sort_values(by=['average'])

In [ ]:
df_correlations = df_correlations.drop(['average'], axis=1)
df_pvalues = df_pvalues.drop(['average'], axis=1)

In [ ]:
df_correlations = df_correlations.T

In [ ]:
fig, ax = plt.subplots(figsize=(6, 12))

def pvalue_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.1:
        return "*"
    else:
        return ""
    
annot = df_correlations.copy().astype(str)

for i in df_correlations.index:
    for j in df_correlations.columns:
        corr = df_correlations.loc[i, j]
        pval = df_pvalues.loc[i, j]
        annot.loc[i, j] = f"{corr:.2f}{pvalue_stars(pval)}"

sns.heatmap(df_correlations, annot=annot, fmt="", cmap="BrBG", linewidths=0.5, ax=ax, 
            annot_kws={"fontsize": 17, "fontproperties": font})

# ax.tick_params(axis='both', which='both', width=0, length=0, labelsize=18, pad=6)

ax.tick_params(axis="both", which="both", bottom=False, left=False, labelbottom=False, labelleft=False)

for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(font)
    label.set_fontsize(18)
    

    
plt.savefig('../plots/correlations/correlations_nolabels.pdf', bbox_inches='tight')

plt.show()